In [4]:
# Installa torchinfo se non disponibile
import importlib, subprocess, sys
if importlib.util.find_spec("torchinfo") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchinfo", "-q"])
print("torchinfo OK")

torchinfo OK


# M04 — Riepilogo dei Modelli

Conteggio parametri, architettura e stime di memoria per:
- **UNetBaseline** — encoder-decoder con skip connections
- **UNetAttention** — stessa baseline con CBAM nel decoder

Questo notebook è puramente documentativo: non esegue training.
I valori qui riportati vanno inclusi nel report (sezione Architettura).

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/input/low-light-repo") if Path("/kaggle").exists() else Path("..")
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from torchinfo import summary
from src.models import UNetBaseline, UNetAttention

print("Import OK")

Import OK


In [6]:
def param_table(model, name):
    """Stampa una tabella compatta di conteggio parametri."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(f"  Parametri totali     : {total:>12,}")
    print(f"  Parametri trainable  : {trainable:>12,}")
    print(f"  Parametri frozen     : {frozen:>12,}")
    print(f"  Dimensione (~float32): {total * 4 / 1024**2:>10.1f} MB")
    return total


## 1. Conteggio parametri

In [7]:
baseline  = UNetBaseline(base_channels=32)
attention = UNetAttention(base_channels=32)

n_base = param_table(baseline,  "UNetBaseline  (base_channels=32)")
n_att  = param_table(attention, "UNetAttention (base_channels=32)")

overhead = (n_att - n_base) / n_base * 100
print(f"\n  Overhead CBAM: +{n_att - n_base:,} parametri ({overhead:.2f}%)")

# Variante con base_channels=64 (per riferimento)
baseline64 = UNetBaseline(base_channels=64)
param_table(baseline64, "UNetBaseline  (base_channels=64)")


──────────────────────────────────────────────────
  UNetBaseline  (base_channels=32)
──────────────────────────────────────────────────
  Parametri totali     :    7,763,107
  Parametri trainable  :    7,763,107
  Parametri frozen     :            0
  Dimensione (~float32):       29.6 MB

──────────────────────────────────────────────────
  UNetAttention (base_channels=32)
──────────────────────────────────────────────────
  Parametri totali     :    7,774,379
  Parametri trainable  :    7,774,379
  Parametri frozen     :            0
  Dimensione (~float32):       29.7 MB

  Overhead CBAM: +11,272 parametri (0.15%)

──────────────────────────────────────────────────
  UNetBaseline  (base_channels=64)
──────────────────────────────────────────────────
  Parametri totali     :   31,037,763
  Parametri trainable  :   31,037,763
  Parametri frozen     :            0
  Dimensione (~float32):      118.4 MB


31037763

## 2. Architettura dettagliata (torchinfo)

Input: `(1, 3, 256, 256)` — batch size 1, RGB, 256×256.

In [8]:
print("=" * 60)
print("  UNetBaseline — architettura")
print("=" * 60)
summary(
    baseline,
    input_size=(1, 3, 256, 256),
    col_names=["input_size", "output_size", "num_params"],
    depth=3,
    verbose=1,
)

  UNetBaseline — architettura
Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
UNetBaseline                                  [1, 3, 256, 256]          [1, 3, 256, 256]          --
├─DoubleConv: 1-1                             [1, 3, 256, 256]          [1, 32, 256, 256]         --
│    └─Sequential: 2-1                        [1, 3, 256, 256]          [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-1                       [1, 3, 256, 256]          [1, 32, 256, 256]         864
│    │    └─BatchNorm2d: 3-2                  [1, 32, 256, 256]         [1, 32, 256, 256]         64
│    │    └─ReLU: 3-3                         [1, 32, 256, 256]         [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-4                       [1, 32, 256, 256]         [1, 32, 256, 256]         9,216
│    │    └─BatchNorm2d: 3-5                  [1, 32, 256, 256]         [1, 32, 256, 256]         64
│    │    └─ReLU: 3-6                         [1, 32

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
UNetBaseline                                  [1, 3, 256, 256]          [1, 3, 256, 256]          --
├─DoubleConv: 1-1                             [1, 3, 256, 256]          [1, 32, 256, 256]         --
│    └─Sequential: 2-1                        [1, 3, 256, 256]          [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-1                       [1, 3, 256, 256]          [1, 32, 256, 256]         864
│    │    └─BatchNorm2d: 3-2                  [1, 32, 256, 256]         [1, 32, 256, 256]         64
│    │    └─ReLU: 3-3                         [1, 32, 256, 256]         [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-4                       [1, 32, 256, 256]         [1, 32, 256, 256]         9,216
│    │    └─BatchNorm2d: 3-5                  [1, 32, 256, 256]         [1, 32, 256, 256]         64
│    │    └─ReLU: 3-6                         [1, 32, 256, 256]         [1, 32, 25

In [9]:
print("=" * 60)
print("  UNetAttention — architettura")
print("=" * 60)
summary(
    attention,
    input_size=(1, 3, 256, 256),
    col_names=["input_size", "output_size", "num_params"],
    depth=3,
    verbose=1,
)

  UNetAttention — architettura
Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
UNetAttention                                 [1, 3, 256, 256]          [1, 3, 256, 256]          --
├─DoubleConv: 1-1                             [1, 3, 256, 256]          [1, 32, 256, 256]         --
│    └─Sequential: 2-1                        [1, 3, 256, 256]          [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-1                       [1, 3, 256, 256]          [1, 32, 256, 256]         864
│    │    └─BatchNorm2d: 3-2                  [1, 32, 256, 256]         [1, 32, 256, 256]         64
│    │    └─ReLU: 3-3                         [1, 32, 256, 256]         [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-4                       [1, 32, 256, 256]         [1, 32, 256, 256]         9,216
│    │    └─BatchNorm2d: 3-5                  [1, 32, 256, 256]         [1, 32, 256, 256]         64
│    │    └─ReLU: 3-6                         [1, 3

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
UNetAttention                                 [1, 3, 256, 256]          [1, 3, 256, 256]          --
├─DoubleConv: 1-1                             [1, 3, 256, 256]          [1, 32, 256, 256]         --
│    └─Sequential: 2-1                        [1, 3, 256, 256]          [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-1                       [1, 3, 256, 256]          [1, 32, 256, 256]         864
│    │    └─BatchNorm2d: 3-2                  [1, 32, 256, 256]         [1, 32, 256, 256]         64
│    │    └─ReLU: 3-3                         [1, 32, 256, 256]         [1, 32, 256, 256]         --
│    │    └─Conv2d: 3-4                       [1, 32, 256, 256]         [1, 32, 256, 256]         9,216
│    │    └─BatchNorm2d: 3-5                  [1, 32, 256, 256]         [1, 32, 256, 256]         64
│    │    └─ReLU: 3-6                         [1, 32, 256, 256]         [1, 32, 25

## 3. Shape delle feature map per livello

Verifica visiva che encoder e decoder abbiano le dimensioni attese.

In [10]:
# Hook per registrare le shape intermedie
shapes = {}

def make_hook(name):
    def hook(module, inp, out):
        shapes[name] = tuple(out.shape)
    return hook

m = UNetBaseline(base_channels=32)
m.inc.register_forward_hook(make_hook("enc0 (inc)"))
m.down1.register_forward_hook(make_hook("enc1 (down1)"))
m.down2.register_forward_hook(make_hook("enc2 (down2)"))
m.down3.register_forward_hook(make_hook("enc3 (down3)"))
m.down4.register_forward_hook(make_hook("enc4 (down4/bottleneck)"))
m.up1.register_forward_hook(make_hook("dec1 (up1)"))
m.up2.register_forward_hook(make_hook("dec2 (up2)"))
m.up3.register_forward_hook(make_hook("dec3 (up3)"))
m.up4.register_forward_hook(make_hook("dec4 (up4)"))
m.outc.register_forward_hook(make_hook("output (outc)"))

x = torch.rand(1, 3, 256, 256)
with torch.no_grad():
    _ = m(x)

print(f"{'Layer':<30} {'Shape (B, C, H, W)'}")
print("-" * 55)
print(f"{'input':<30} {tuple(x.shape)}")
for name, shape in shapes.items():
    print(f"{name:<30} {shape}")


Layer                          Shape (B, C, H, W)
-------------------------------------------------------
input                          (1, 3, 256, 256)
enc0 (inc)                     (1, 32, 256, 256)
enc1 (down1)                   (1, 64, 128, 128)
enc2 (down2)                   (1, 128, 64, 64)
enc3 (down3)                   (1, 256, 32, 32)
enc4 (down4/bottleneck)        (1, 512, 16, 16)
dec1 (up1)                     (1, 256, 32, 32)
dec2 (up2)                     (1, 128, 64, 64)
dec3 (up3)                     (1, 64, 128, 128)
dec4 (up4)                     (1, 32, 256, 256)
output (outc)                  (1, 3, 256, 256)


## 4. Stima consumo VRAM

> Le stime sono **indicative**: il consumo reale dipende da framework overhead,
> gradienti, ottimizzatore e AMP. Verificare sempre con `torch.cuda.memory_summary()`
> durante il training reale.

In [11]:
def estimate_vram(model, batch_size, h=256, w=256, amp=True):
    """
    Stima VRAM in MB per un singolo forward+backward.

    Componenti considerate:
      - Parametri + gradienti (x2 se training)
      - Stato ottimizzatore Adam (x2 per momento 1 e 2)
      - Attivazioni input batch
    AMP (float16) dimezza il consumo delle attivazioni.
    """
    bytes_per_param = 4  # float32
    n_params = sum(p.numel() for p in model.parameters())

    # Parametri + gradienti
    param_mb = n_params * bytes_per_param * 2 / 1024**2
    # Stato ottimizzatore Adam (m1 + m2, float32)
    optim_mb = n_params * bytes_per_param * 2 / 1024**2
    # Attivazioni input (stima grossolana: ~8 byte/px con AMP, ~16 senza)
    act_bytes = 2 if amp else 4
    act_mb = batch_size * 3 * h * w * act_bytes / 1024**2

    total = param_mb + optim_mb + act_mb
    return param_mb, optim_mb, act_mb, total


print(f"{'Modello':<25} {'bs':>4} {'AMP':>5}  "
      f"{'Params MB':>9} {'Optim MB':>9} {'Act MB':>7} {'Totale MB':>9}")
print("-" * 70)

configs = [
    (baseline,          "UNetBaseline",  8,  True),
    (baseline,          "UNetBaseline",  8,  False),
    (baseline,          "UNetBaseline",  16, True),
    (attention,         "UNetAttention", 8,  True),
    (attention,         "UNetAttention", 16, True),
    (baseline64,        "Baseline x64",  8,  True),
]

for model, name, bs, amp in configs:
    p, o, a, tot = estimate_vram(model, bs, amp=amp)
    amp_str = "fp16" if amp else "fp32"
    print(f"{name:<25} {bs:>4} {amp_str:>5}  "
          f"{p:>9.1f} {o:>9.1f} {a:>7.1f} {tot:>9.1f}")

print()
print("Nota: la T4 Kaggle ha 16 GB VRAM. Tutte le config con AMP sono compatibili.")


Modello                     bs   AMP  Params MB  Optim MB  Act MB Totale MB
----------------------------------------------------------------------
UNetBaseline                 8  fp16       59.2      59.2     3.0     121.5
UNetBaseline                 8  fp32       59.2      59.2     6.0     124.5
UNetBaseline                16  fp16       59.2      59.2     6.0     124.5
UNetAttention                8  fp16       59.3      59.3     3.0     121.6
UNetAttention               16  fp16       59.3      59.3     6.0     124.6
Baseline x64                 8  fp16      236.8     236.8     3.0     476.6

Nota: la T4 Kaggle ha 16 GB VRAM. Tutte le config con AMP sono compatibili.


## 5. Tabella riepilogativa per il report

| Modello | base_channels | Parametri | Modello (MB) | CBAM overhead |
|---|---|---|---|---|
| UNetBaseline | 32 | ~7.76 M | ~29.6 MB | — |
| UNetAttention | 32 | ~7.88 M | ~30.1 MB | < 2% |
| UNetBaseline | 64 | ~31.0 M | ~118.5 MB | — |

> I valori esatti sono quelli stampati nelle celle precedenti.
> Aggiornare questa tabella se si cambia `base_channels`.